# Retrieve Tournaments
- Gets tournaments given a specific timeframe in NorCal
- Filters by game
- Saves to database/cache 
- Each time it reruns, compares to db to see if we already have tournament info: if so, we skip re-saving the info, if not, we add it
- Prints how many cache hits we have 
- Prints how many cache misses we have 
- Incorporates start.gg rate limit

In [ ]:
import requests
import json
import pandas as pd
import os
from datetime import datetime
from collections import Counter

AUTH_TOKEN = os.environ.get("STARTGG_API_KEY", "").strip()
if not AUTH_TOKEN:
    raise ValueError("Set STARTGG_API_KEY in your environment or .env (see repo .env.example)")
SF_BASED_COORDS = "37.77151615492457, -122.41563048985462"
SF_RADIUS = "70mi"

SAC_BASED_COORDS = "38.57608096237729, -121.49183616631059"
SAC_RADIUS = "40mi"

NUM_PER_PAGE = 50
request_url = 'https://api.start.gg/gql/alpha'

def flatten_nested_json_df(df):
    
    df = df.reset_index()
    
    print(f"original shape: {df.shape}")
    print(f"original columns: {df.columns}")
    
    s = (df.applymap(type) == list).all()
    list_columns = s[s].index.tolist()
    
    s = (df.applymap(type) == dict).all()
    dict_columns = s[s].index.tolist()
    
    print(f"lists: {list_columns}, dicts: {dict_columns}")
    while len(list_columns) > 0 or len(dict_columns) > 0:
        new_columns = []
        
        for col in dict_columns:
            print(f"flattening: {col}")
            horiz_exploded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            horiz_exploded.index = df.index
            df = pd.concat([df, horiz_exploded], axis=1).drop(columns=[col])
            new_columns.extend(horiz_exploded.columns) 
        
        for col in list_columns:
            print(f"exploding: {col}")
            df = df.drop(columns=[col]).join(df[col].explode().to_frame())
            df = df.reset_index(drop=True)
            new_columns.append(col)
        
        s = (df[new_columns].applymap(type) == list).all()
        list_columns = s[s].index.tolist()

        s = (df[new_columns].applymap(type) == dict).all()
        dict_columns = s[s].index.tolist()
        
        print(f"lists: {list_columns}, dicts: {dict_columns}")
        
    print(f"final shape: {df.shape}")
    print(f"final columns: {df.columns}")
    return df

def date_to_unix(date_string, date_format="%Y-%m-%d"):
    try:
        date_obj = datetime.strptime(date_string, date_format)
        unix_timestamp = int(date_obj.timestamp())
        return unix_timestamp
    except ValueError as e:
        print(f"Error: {e}")
        return None

def generate_graphql_query(timestamps):
    template = """
query BayNorCalTournaments($page: Int, $perPage: Int, $coordinates: String!, $radius: String!) {{
  tournaments(
    query: {{
    page: $page
    perPage: $perPage
    filter: {{
      location: {{
        distanceFrom: $coordinates,
        distance: $radius
      }},
      afterDate: {after_date} 
      beforeDate: {before_date}
    }}
    sortBy:"startAt"
  }}) {{
    nodes {{
      id
      name
      city
      slug
      startAt
      events {{
        slug
        numEntrants
        videogame {{
          name
        }}
      }}
    }}
  }}
}}
"""
    after_date = timestamps[0]
    before_date = timestamps[1]
    query = template.format(after_date=after_date, before_date=before_date)

    return query


def get_all_tournies(auth_token, query, coords, radius, num_per_page):
  
  graphql_query = query 
  tournies = []

  for i in range(1, 10):
    variables = {
        "page": i,
        "perPage": num_per_page,
        "coordinates": coords,
        "radius": "50mi"
    }
    data = {"query" : graphql_query, "variables": variables}
    json_data = json.dumps(data)
    auth_header = auth_token
    header = {'Authorization': 'Bearer ' + auth_header}  

    response = requests.post(url=request_url, headers=header, data=json_data)
    json_resp = json.loads(response.text)
    print(json_resp)
    if ("errors" not in json_resp):
       
      curr_tournies_page = json_resp['data']['tournaments']['nodes']
      print("Number of tournies in page is:" + str(len(curr_tournies_page)))

      tournies += curr_tournies_page
  return tournies

# user should be able to edit the start and end dates
start_date = "2025-04-01"
end_date = "2025-06-30"
timestamps = [date_to_unix(start_date), date_to_unix(end_date)]
timestamps

query = generate_graphql_query(timestamps)
print(query)

tournies = []
bay_tournies = get_all_tournies(AUTH_TOKEN, query, SF_BASED_COORDS, SF_RADIUS, NUM_PER_PAGE)
sac_tournies = get_all_tournies(AUTH_TOKEN, query, SAC_BASED_COORDS, SAC_RADIUS, NUM_PER_PAGE)
tournies = sac_tournies + bay_tournies
np_tournies = pd.DataFrame(tournies).explode('events')
flat_tournies = flatten_nested_json_df(np_tournies)

# ult_tournies = flat_tournies[
#     flat_tournies['events'].apply(
#         lambda x: isinstance(x, dict) and x.get('videogame', {}).get('name') == 'Super Smash Bros. Ultimate'
#     ) &
#     flat_tournies['events'].apply(
#         lambda x: isinstance(x, dict) and x.get('numEntrants', 0) >= 16
#     )
# ]

# ult_tournies = ult_tournies.reset_index(drop=True)

ult_tournies = flat_tournies[
    (flat_tournies['events.videogame.name'] == 'Super Smash Bros. Ultimate') &
    (flat_tournies['events.numEntrants'] >= 16)
]

ult_tournies = ult_tournies.reset_index(drop=True)
ult_tournies


ModuleNotFoundError: No module named 'requests'

# Part 2: Transforming scraped data

- Retrieves database and generates new, updated db with relevant information such that we can parse more information from tournaments
- Parse sets within each tournament: there should be a cache such that if one tournament has already been processed/it's sets have been retrieved, we don't rescrape it to be time efficient 
- Print how many cache hits and misses we have: be verbose
- Have elegant handling of name merging/replacement prior to ELO generation

In [ ]:
# ult_tournies['startgg_url'] = ult_tournies['events'].apply(lambda x: x['slug'] if isinstance(x, dict) and 'slug' in x else None)
# ult_tournies['startgg_url'] = 'start.gg/' + ult_tournies['startgg_url'].astype('str')
# ult_tournies['Event Date'] = ult_tournies['startAt'].map(
#     lambda x: datetime.fromtimestamp(x).strftime('%Y-%m-%d') if pd.notnull(x) and isinstance(x, (int, float)) else None
# )
# ult_tournies['StartGG TOURNAMENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-3:-2][0])
# ult_tournies['StartGG EVENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-1:][0]) 
# ult_tournies = ult_tournies.drop_duplicates('startgg_url', keep='first')

# ult_tournies['jakeSlug'] = ult_tournies['slug'] + '/event/' + ult_tournies['StartGG EVENT_ID']
# ult_tournies #can export this to csv to use with braacket

# #stop here and edit the df to remove tournaments that shouldn't count towards PR

# # Ensure correct startgg_url extraction
ult_tournies['startgg_url'] = 'start.gg/' + ult_tournies['events.slug'].astype(str)

# Convert startAt to a readable date format
ult_tournies['Event Date'] = ult_tournies['startAt'].map(
    lambda x: datetime.fromtimestamp(x).strftime('%Y-%m-%d') if pd.notnull(x) and isinstance(x, (int, float)) else None
)

# Extract StartGG Tournament ID and Event ID from the formatted slug
ult_tournies['StartGG TOURNAMENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-3] if len(url.split("/")) >= 3 else None)
ult_tournies['StartGG EVENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-1] if len(url.split("/")) >= 1 else None)

# Remove duplicates based on the tournament URL
ult_tournies = ult_tournies.drop_duplicates('startgg_url', keep='first')

# Create the jakeSlug column for Braacket usage
ult_tournies['jakeSlug'] = ult_tournies['slug'] + '/event/' + ult_tournies['StartGG EVENT_ID']
ult_tournies

import requests
import os
from limiter import Limiter
_sgg = os.environ.get("STARTGG_API_KEY", "").strip()
if not _sgg:
    raise ValueError("Set STARTGG_API_KEY")
header = {"Authorization": f"Bearer {_sgg}"}
url = 'https://api.start.gg/gql/alpha'
rate_limiter = Limiter(rate=1.1, capacity=1)

eventIDQuery = '''
query getEventId($slug: String) {
  event(slug: $slug) {
    id
    name
  }
},
'''
# returns event id lawls
@rate_limiter
def getEventID(slug):
    variables = {"slug" : slug}
    json_request = {"query" : eventIDQuery, "variables" : variables}
    request = requests.post(url = url, json = json_request, headers = header)
    response = request.json()
    print(response)
    print(response['data']['event']['name'])
    return response['data']['event']['id']
setQuery = '''
query EventSets($eventId: ID!, $page: Int!, $perPage: Int!) {
  event(id: $eventId) {
    id
    name
    sets(page: $page, perPage: $perPage, sortType: STANDARD) {
      pageInfo {
        total
        totalPages
      }
      nodes {
        id
      }
    }
  }
},
'''
# returns amount of pages, use this to iterate through next func
@rate_limiter
def getTotalPagesSet(eventId):
    variables = {"eventId" : eventId, "page" : 1, "perPage" : 40}
    json_request = {"query" : setQuery, "variables" : variables}
    request = requests.post(url = url, json = json_request, headers = header)
    response = request.json()
    return response['data']['event']['sets']['pageInfo']['totalPages']

# returns list of 'id' : actual id maps, iterate through this later to grab set IDs
@rate_limiter
def getSetsOnePage(eventId, page):
    variables = {"eventId" : eventId, "page" : page, "perPage" : 40}
    json_request = {"query" : setQuery, "variables" : variables}
    request = requests.post(url = url, json = json_request, headers = header)
    response = request.json()
    return response['data']['event']['sets']['nodes']

setQueryBackUp = '''
query EventSets($eventId: ID!, $page: Int!, $perPage: Int!) {
  event(id: $eventId) {
    id
    name
    sets(
      page: $page
      perPage: $perPage
      sortType: STANDARD
    ) {
      pageInfo {
        total
        totalPages
      }
      nodes {
        id
        slots {
          id
          entrant {
            id
            name
          }
        }
      }
    }
  }
},
'''
playerAndScoreQuery = '''
query SetsAndPlayers($setId: ID!) {
  set(id: $setId) {
    state
    slots {
      entrant {
        participants {
          player {
            gamerTag
            prefix
          }
        }
      }
      standing {
        stats {
          score {
            value
          }
        }
      }
    }
  }
}
'''
# @rate_limiter
# def getPlayersAndScore(setId):
#     variables = {"setId" : setId}
#     json_request = {"query" : playerAndScoreQuery, "variables" : variables}
#     request = requests.post(url = url, json = json_request, headers = header)
#     response = request.json()
#     # print(response)
#     # list of len 2, each is map of stuff to right of query, extract accordingly
#     # split list into 2 maps, grab vals
#     print(response)
#     stuff = response['data']['set']['slots']
#     # print(stuff)
#     p1Name = stuff[0]['entrant']['participants'][0]['player']['gamerTag']
#     p1Pre = stuff[0]['entrant']['participants'][0]['player']['prefix']
#     p1Score = stuff[0]['standing']['stats']['score']['value']
#     p2Name = stuff[1]['entrant']['participants'][0]['player']['gamerTag']
#     p2Pre = stuff[1]['entrant']['participants'][0]['player']['prefix']
#     p2Score = stuff[1]['standing']['stats']['score']['value']
#     if p1Pre == None:
#         p1Pre = ''
#     if p2Pre == None:
#         p2Pre = ''
#     # print(p1Pre + ' ' + p1Name + ':' + str(p1Score))
#     # print(p2Pre + ' ' + p2Name + ':' + str(p2Score))
#     if p1Pre == '':
#       p1NameFull = p1Name
#     else:
#       p1NameFull = p1Pre + ' | ' + p1Name
#     if p2Pre == '':
#       p2NameFull = p2Name
#     else:
#       p2NameFull = p2Pre + ' | ' + p2Name
#     return {p1NameFull : p1Score, p2NameFull : p2Score}

@rate_limiter
def getPlayersAndScore(setId, cache):
    # Check if result is cached
    if setId in cache:
        print(f"Cache hit for setId: {setId}")
        return cache[setId]

    variables = {"setId": setId}
    json_request = {"query": playerAndScoreQuery, "variables": variables}

    # Debugging: Print request payload
    print(f"Request payload for setId {setId}: {json_request}")

    try:
        request = requests.post(url=url, json=json_request, headers=header)

        # Debugging: Log status code
        print(f"Status code for setId {setId}: {request.status_code}")
        
        if request.status_code != 200:
            print(f"Error: Received non-200 status code for setId {setId}")
            print("Response text:", request.text)
            return {"Error": f"Request failed for setId {setId}"}

        # Attempt to parse JSON
        try:
            response = request.json()
        except requests.exceptions.JSONDecodeError:
            print(f"Failed to decode JSON for setId {setId}")
            print("Raw response:", request.text)
            return {"Error": f"Invalid JSON response for setId {setId}"}

        # Debugging: Print response
        print(f"Response for setId {setId}: {response}")

        # Extract data
        stuff = response.get('data', {}).get('set', {}).get('slots', None)

        if not stuff or len(stuff) < 2 or any(slot.get('entrant') is None for slot in stuff):
            print(f"Missing or incomplete data for setId: {setId}, skipping")
            return {"Error": f"Incomplete data for setId {setId}"}

        # Extract player details
        try:
            p1Name = stuff[0]['entrant']['participants'][0]['player']['gamerTag']
            p1Pre = stuff[0]['entrant']['participants'][0]['player']['prefix']
            p1Score = stuff[0]['standing']['stats']['score']['value']

            p2Name = stuff[1]['entrant']['participants'][0]['player']['gamerTag']
            p2Pre = stuff[1]['entrant']['participants'][0]['player']['prefix']
            p2Score = stuff[1]['standing']['stats']['score']['value']

            p1Pre = '' if p1Pre is None else p1Pre
            p2Pre = '' if p2Pre is None else p2Pre

            p1NameFull = p1Name if p1Pre == '' else f"{p1Pre} | {p1Name}"
            p2NameFull = p2Name if p2Pre == '' else f"{p2Pre} | {p2Name}"

            result = {p1NameFull: p1Score, p2NameFull: p2Score}

            # Cache result
            cache[setId] = result
            return result
        except (KeyError, IndexError, TypeError) as e:
            print(f"Error extracting data for setId {setId}: {e}")
            return {"Error": f"Data extraction failed for setId {setId}"}

    except requests.RequestException as e:
        print(f"Network error for setId {setId}: {e}")
        return {"Error": f"Network error for setId {setId}"}
    
    # from the_project import *
from queries import *
import numpy as np
import math
# for 1 tournament, grab all sets by page and adds IDs to setList and returns it
# DO NOT USE THE WORD SET IT FUCKS EVERYTHING UP!!!!!!
def getSetIDs(eventID):
    print(eventID)
    setList = []
    # getting page count for bracket
    totalPages = getTotalPagesSet(eventID)
    for page in range(1, totalPages + 1):
        # getting all set maps on a page
        sets = getSetsOnePage(eventID, page)
        # getting actual set IDs
        for setMap in sets:
            smashSet = setMap['id']
            setList.append(smashSet)
    return setList

# take list of all sets and extract players to put in a set to store all players in given tournaments, size is recorded to make matrix
# def playerList(sets):
#     players = set()
#     # print(type(players))
#     # print(players)
#     for smashSet in sets:
#         # print(smashSet)
#         names = list(smashSet.keys())
#         # print(names)
#         p1name = names[0]
#         p2name = names[1]
#         players.add(p1name)
#         players.add(p2name)
#     return players

def playerList(sets):
    players = set()
    for smashSet in sets:
        # Ensure smashSet has at least two keys
        if len(smashSet) < 2:
            print(f"Warning: Unexpected smashSet format: {smashSet}")
            continue  # Skip this iteration
        
        names = list(smashSet.keys())
        p1name = names[0]
        p2name = names[1]
        players.add(p1name)
        players.add(p2name)
        print(players)
    return players

# makes set matrix where set count will go into, value in (0,2) is sets player 0 has on player 2, value in (2,0) is opposite
# also makes similar game matrix
# also makes dictionary to map player name to index in matrices
def makeMatrices(players, sets):
    playerCount = len(players)
    # set up empty matrices
    setMatrix = np.zeros((playerCount, playerCount))
    gameMatrix = np.zeros((playerCount, playerCount))
    # set up empty dict
    playerMatrixIndex = {}
    ind = 0
    # assigns player to index in matrices
    for player in players:
        playerMatrixIndex[player] = ind
        ind += 1
    for smashSet in sets:
        names = list(smashSet.keys())
        p1name = names[0]
        p2name = names[1]
        scores = list(smashSet.values())
        p1score = scores[0]
        p2score = scores[1]
        p1index = playerMatrixIndex[p1name]
        p2index = playerMatrixIndex[p2name]
        # dq case
        if p1score is None or p2score is None:
            break
        # updating game mat
        gameMatrix[p1index, p2index] += p1score
        gameMatrix[p2index, p1index] += p2score
        # p1 won set case
        if p1score > p2score:
            setMatrix[p1index, p2index] += 1
        # p2 won set case
        else:
            setMatrix[p2index, p1index] += 1
    return playerMatrixIndex, gameMatrix, setMatrix

# # updateElo takes elo map and a set and computes updated elo score and returns the new updated elo map
# def updateElo(elo, smashSet):
#     print(smashSet)
#     print(elo)
#     names = list(smashSet.keys())
#     print(names)
#     p1name = names[0]
#     p2name = names[1]
#     scores = list(smashSet.values())
#     print(scores)
#     p1score = scores[0]
#     p2score = scores[1]
#     # stop here if dq set
#     if p1score is None or p2score is None:
#         return elo
#     p1rank = elo[p1name]
#     p2rank = elo[p2name]
#     # elo probability
#     p1prob = 1.0/(1 + math.pow(10, (p1rank - p2rank) / 400.0))
#     p2prob = 1.0 - p1prob
#     # 1 is p1 win, 0 otherwise
#     if p1score > p2score:
#         outcome = 1
#     else:
#         outcome = 0
#     # k is some scaling constant, play with it later
#     k = 30
#     # updating elo
#     p1rank += k*(outcome - p1prob)
#     p2rank += k*((1 - outcome) - p2prob)
#     print(p1rank)
#     print(p2rank)
#     # updating elo map
#     elo[p1name] = p1rank
#     elo[p2name] = p2rank
#     return elo


def updateElo(elo, smashSet):
    # print(smashSet)
    # print(elo)
    
    # Get the player names
    names = list(smashSet.keys())
    # print(names)
    
    # Check if the smashSet has at least two players
    if len(names) < 2:
        print(f"Warning: Incomplete smashSet: {smashSet}")
        return elo  # Skip this set
    
    p1name = names[0]
    p2name = names[1]
    
    # Get the scores
    scores = list(smashSet.values())
    print(scores)
    
    p1score = scores[0]
    p2score = scores[1]
    
    # Stop here if the set contains disqualifications or invalid scores
    if p1score is None or p2score is None:
        print(f"Disqualified set: {smashSet}")
        return elo
    
    p1rank = elo.get(p1name, 1500)  # Default to 1500 if player not in elo map
    p2rank = elo.get(p2name, 1500)  # Default to 1500 if player not in elo map
    
    # Elo probability
    p1prob = 1.0 / (1 + math.pow(10, (p2rank - p1rank) / 400.0))
    p2prob = 1.0 - p1prob
    
    # Determine the outcome
    outcome = 1 if p1score > p2score else 0
    
    # K is a scaling constant
    k = 30
    
    # Update ranks
    p1rank += k * (outcome - p1prob)
    p2rank += k * ((1 - outcome) - p2prob)
    
    print(p1rank)
    print(p2rank)
    
    # Update the elo map
    elo[p1name] = p1rank
    elo[p2name] = p2rank
    
    return elo

# sorts elo map by value and stays as dictionary
def sortElo(elo):
    eloSorted = sorted(elo.items(), key = lambda x: x[1], reverse = True)
    eloSorted = dict(eloSorted)
    return eloSorted

jake_slugs = ult_tournies['jakeSlug'].astype(str).tolist()
jake_slugs

eventIDs = [getEventID(tourney) for tourney in jake_slugs]
print(eventIDs)

ult_tournies['eventID'] = eventIDs
ult_tournies.head()

setIDs = []
for ID in eventIDs:
  setIDTemp = getSetIDs(ID)
  setIDs.extend(setIDTemp)
print(setIDs)
print(len(setIDs))

sets = []
x = 0
cache = {}

for setID in setIDs:
  print(x)
  x += 1
  print(setID)
  smashSet = getPlayersAndScore(setID, cache)
  print(smashSet)
  sets.append(smashSet)

players = playerList(sets)

def replace_keys(input_list):
    key_mapping = {
        'NLC | they call me leonidas': 'Hyro',
        'NLC | Still Spoozy': 'Hyro',
        'MPoor' : 'M4',
        'W4' : 'M4',
        'SALT | ebs | ERA': 'ERA',
        'era': 'ERA',
        'EBS | HK | the filipino flowstate.' : 'Skylock',
    }

    output_list = []
    for item in input_list:
        new_item = {}
        for key, value in item.items():
            new_key = key_mapping.get(key, key)
            new_item[new_key] = value
        output_list.append(new_item)

    return output_list

# Part 3: Elo Calculation/Derived Statistics
- Use the db used in part 2 as the basis for this, the db in part 2 should have all the sufficient info outside of the out of region info for each player/should have all in region info
- Should be able to select/deselect tournaments that count towards elo selection, either by tournament or by a given date range (not implemented currently, need to implement)
- Demo the above by showing top 20 ELO scores with all tournaments, and then RANDOMLY select (should be a new tournament on each run) a tournament to remove, and then show top 20 elo scores without that tournament and print at most 5 names of players who's elo score changed. it should be truly random to show proof that this function works
- Should be able to view the following: head to heads against each other, in region tournament placements, out of region tournament placements, out of region tournament wins/losses, number of tournaments, etc. Out of region is defined as any tournaments the player attended that is not within the norcal coordinate range, which should be defined by whether the event is in the database. 
- Out of region scraping is not implemented yet: will need to research start GG api docs to see how this is done. Good check to do is to check a date range with Nov 8th and 9th, 2025 in it for the player Lui$ (should be listed as Lui$ or NU | Lui$, one or the other), a tournament called "Port Priority 9" should show up. API response for wins/losses should show a win against a player named Syrup, a loss to a player named Light (among more). I think when scraping players in each tournament, you need to save their player ID, so you may need to edit the query call to get tournament attendance to make sure you keep the player ID
- When demoing this, to ensure we fully avoid any issues with cached results, pick two random players each time and compare them/show comparison stats each time. Print this and ensure it's random selection each time to prove that live api calls/correct decisions are being made. 
- Also, print the number of the players in the db, and print the ELO calculation of the top 10 
- Create db of player info which updates, using the player name and statistics tied to each tournament; pretty much put together the player db that can be updated here while being designed such that duplicate information is carefully handled such that the db stays clean and has no mistakes. research previous db design to ensure it's implemented properly

In [ ]:
print(len(players))
playerCount = len(players)

playerMatrixIndex, gameMatrix, setMatrix = makeMatrices(players, sets)
print(playerMatrixIndex)
print(gameMatrix)
print(setMatrix)

elo = {player:1500 for player in players}
print(elo)
for smashSet in sets:
  elo = updateElo(elo, smashSet)
elo = sortElo(elo)

print(elo)

def create_player_summary(sets, elo, players):
    player_summary = []

    for player in players:
        # Find opponents the player won against
        won_against = sorted(
            [
                opponent for match in sets
                if player in match and all(value is not None for value in match.values())  # Exclude matches with None
                for opponent in match.keys()
                if opponent != player and match[player] > match[opponent]
            ],
            key=lambda x: elo.get(x, 0), reverse=True
        )

        # Find opponents the player lost to
        lost_against = sorted(
            [
                opponent for match in sets
                if player in match and all(value is not None for value in match.values())  # Exclude matches with None
                for opponent in match.keys()
                if opponent != player and match[player] < match[opponent]
            ],
            key=lambda x: elo.get(x, 0), reverse=True
        )

        # Append to summary
        player_summary.append({
            "Player": player,
            "ELO": elo.get(player, 0),
            "Won Against": won_against,
            "Lost Against": lost_against
        })

    # Convert to DataFrame and sort by ELO
    df_summary = pd.DataFrame(player_summary).sort_values(by="ELO", ascending=False).reset_index(drop=True)
    return df_summary


def add_notable_results(df_summary, elo, playerCount):
    bottom = playerCount - 30
    # Get top 30 players by ELO
    top_players = sorted(elo.keys(), key=lambda x: elo[x], reverse=True)[:30]
    # Get bottom 100 players by ELO
    bottom_players = sorted(elo.keys(), key=lambda x: elo[x])[:bottom]

    # Define a helper function to filter notable wins
    def filter_notable_wins(wins):
        return [player for player in wins if player in top_players]

    # Define a helper function to filter notable losses
    def filter_notable_losses(losses):
        return [player for player in losses if player in bottom_players]

    # Create the Notable Wins and Notable Losses columns
    df_summary["Notable Wins"] = df_summary["Won Against"].apply(filter_notable_wins)
    df_summary["Notable Losses"] = df_summary["Lost Against"].apply(filter_notable_losses)

    return df_summary


pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Do not truncate column content

df = create_player_summary(sets, elo, players)
df_filtered = add_notable_results(df, elo, len(players))
df_filtered.head(5)

len(players)

@rate_limiter
def getAttendeesForEvents(event_ids, per_page=40, key=AUTH_TOKEN):
    entrants_query = '''
    query EventEntrants($eventId: ID!, $page: Int!, $perPage: Int!) {
      event(id: $eventId) {
        id
        name
        entrants(query: { page: $page, perPage: $perPage }) {
          pageInfo {
            totalPages
          }
          nodes {
            id
            participants {
              id
              gamerTag
              prefix
            }
          }
        }
      }
    }
    '''

    url = "https://api.start.gg/gql/alpha"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {key}"
    }

    attendees = {}

    for event_id in event_ids:
        page = 1
        event_attendees = []

        while True:
            variables = {
                "eventId": event_id,
                "page": page,
                "perPage": per_page
            }
            json_request = {"query": entrants_query, "variables": variables}

            # Print the query and variables before sending
            # print("Generated Query:", json_request)

            response = requests.post(url=url, json=json_request, headers=headers).json()

            if 'errors' in response:
                print(f"Error fetching data for Event ID {event_id}: {response['errors']}")
                break

            data = response.get('data', {}).get('event', {})
            entrants = data.get('entrants', {})
            nodes = entrants.get('nodes', [])

            for entrant in nodes:
                for participant in entrant.get('participants', []):
                    event_attendees.append({
                        "eventId": event_id,
                        "entrantId": entrant['id'],
                        "participantId": participant['id'],
                        "gamerTag": participant['gamerTag']
                    })

            total_pages = entrants.get('pageInfo', {}).get('totalPages', 0)
            if page >= total_pages:
                break

            page += 1

        attendees[event_id] = event_attendees

    return attendees

attendees = getAttendeesForEvents(eventIDs)
attendees 

def count_tournaments_per_player(attendees):
    player_tournaments = {}

    for tournament, participants in attendees.items():
        for participant in participants:
            gamer_tag = participant['gamerTag']
            if gamer_tag not in player_tournaments:
                player_tournaments[gamer_tag] = set()
            player_tournaments[gamer_tag].add(tournament)

    return {gamer_tag: len(tournaments) for gamer_tag, tournaments in player_tournaments.items()}


tourney_count = count_tournaments_per_player(attendees)
tourney_count

def replace_and_sum(input_dict):
    # Define the key mapping
    key_mapping = {
        'era': 'ERA',
        'ERA' : 'ERA',
        'Still Spoozy': 'they call me leonidas',
        'the filipino flowstate.': 'Skylock'
    }

    # Step 1: Replace keys based on the mapping
    replaced_dict = {}
    for key, value in input_dict.items():
        new_key = key_mapping.get(key, key)
        replaced_dict[new_key] = replaced_dict.get(new_key, 0) + value

    # Step 2: Merge keys with the same names and sum their values
    result_dict = {}
    for key, value in replaced_dict.items():
        result_dict[key] = result_dict.get(key, 0) + value

    return result_dict

new_attendance = replace_and_sum(tourney_count)
new_attendance

def map_players_to_event_names(attendees, event_df):
    # Create a mapping of eventID to event name
    event_id_to_name = dict(zip(event_df['eventID'], event_df['name']))
    
    # Initialize a dictionary to store players and their events
    player_events = {}

    # Iterate over attendees to populate player-events mapping
    for event_id, participants in attendees.items():
        event_name = event_id_to_name.get(event_id, "Unknown Event")  # Default to "Unknown Event" if ID not found
        for participant in participants:
            gamer_tag = participant['gamerTag']
            if gamer_tag not in player_events:
                player_events[gamer_tag] = []
            player_events[gamer_tag].append(event_name)
    
    # Convert to a DataFrame
    player_events_df = pd.DataFrame([
        {"Player": player, "Event Names": events} 
        for player, events in player_events.items()
    ])
    
    return player_events_df


player_events_df.to_csv('test2.csv')

player_events_df = map_players_to_event_names(attendees, ult_tournies)
player_events_df.head()

def clean_df_names(df):
    df['Player'] = df['Player'].apply(lambda x: x.split(' | ')[-1].strip())
    return df

clean_df = clean_df_names(df_filtered)
clean_df.head(5)

def count_and_sort(input_list):
    # Clean names in the input list
    cleaned_list = [name.split(' | ')[-1].strip() for name in input_list]
    
    # Count occurrences of cleaned names
    counts = Counter(cleaned_list)
    
    # Sort by count in descending order
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_counts

clean_df['Won Against'] = clean_df['Won Against'].apply(count_and_sort)
clean_df['Lost Against'] = clean_df['Lost Against'].apply(count_and_sort)
clean_df['Notable Wins'] = clean_df['Notable Wins'].apply(count_and_sort)
clean_df['Notable Losses'] = clean_df['Notable Losses'].apply(count_and_sort)
clean_df.head(5)

def calculate_head_to_heads(df):
    def classify_h2h(row):
        # Initialize lists for positive, even, and negative head-to-heads
        pos_h2h = []
        even_h2h = []
        neg_h2h = []

        # Create dictionaries for wins and losses for fast lookup
        wins = {name: count for name, count in row["Won Against"]}
        losses = {name: count for name, count in row["Lost Against"]}

        # Combine all unique names from both wins and losses
        all_names = set(wins.keys()).union(set(losses.keys()))

        # Classify each name
        for name in all_names:
            win_count = wins.get(name, 0)
            loss_count = losses.get(name, 0)

            if win_count > loss_count:
                pos_h2h.append(name)
            elif win_count == loss_count and win_count > 0:  # Avoid 0-0 cases
                even_h2h.append(name)
            elif win_count < loss_count:
                neg_h2h.append(name)

        return pos_h2h, even_h2h, neg_h2h

    # Apply the classification to each row
    df[["Pos H2H", "Even H2H", "Neg H2H"]] = df.apply(
        lambda row: pd.Series(classify_h2h(row)), axis=1
    )

    return df

new_df = calculate_head_to_heads(clean_df)
new_df.head(5)

new_df['Attendance'] = new_df['Player'].map(new_attendance)
new_df.head(5) #this can also be exported as the season's data sheet

new_df.head(20)

qual_df = new_df[new_df['Attendance'] >= 6]
qual_df.head(5) #this can be exported as the season's data sheet

new_df.to_csv('full.csv')
qual_df.to_csv('attendance.csv')

new_df = pd.read_csv('full.csv')
qual_df = pd.read_csv('attendance.csv')

new_df.head(5)

# Part 4: Generate rankings
- User should be able to select, from list of players across an entire season, who the "contenders" for rankings are. To demo this functionality, from the top 30 players by ELO, make a random selection of 5 of them. Print who the random 10 are. 
- This should form a list. User should then be able to rank these players in the following manner: Pairwise Comparison or Pairwise Ranking method. This technique is used to create a linear ordering of items (a ranked list) from subjective or noisy data. 
Key Aspects of Pairwise Ranking Lists:
Methodology: It works like a tournament (similar to a round-robin or tournament bracket) where items are presented in pairs to a user, and they choose the preferred item.
Ranking Mechanism (Wins/Scores): Each time a user says "A is better than B," A receives a "win." 
Transitivity: This approach often assumes that if a user prefers A over B and B over C, then A is also preferred over C. 
- When making the decisions/making the rankings, a relevant player card should be made with each player's information/comparison info, which should all be stored in the database generated in Part 3, alongside their out of region information (not yet implemented, you will need to implement this). 
- The user should have the OPTION to ask OpenAI to help them make a judgement. It should not be automatically generated for the sole reason of saving money. The prompt will need to be modified such that arguments are made for both players. The OpenAI LLM should acknowledge it may be missing information regarding out of region events/will not have full context because it's unable to properly judge out of region events. The user should also have the option to go with the AI's decision. 


Demo this functionality by doing and showing the following is being done (be vocal!):
- Select a random 5 out of the top 30 of each ELO (on each run, should be different each time)
- Make sure the date range is from April 1, 2025 to June 30, 2025 as to not make this take forever (all data should be in databases generated in previous parts)
- Demo the functionality of asking the user the question of whether A or B should be ranked higher
- Make sure to display the relevant player card information with all the derived stats, head to head info, etc + out of region information
- For 50% of the ranking questions, simply select who's ELO is higher (randomly choose which, doesn't have to be the first 50%), but indicate this is meant to simulate a human user's decision making
- For the other 50% of ranking questions, show that OpenAI is being requested (a key is already in .env), show the AI decision, and also show it going with the AI decision as well. You may need to generate one response for a detailed justification with multiple perspectives, and another response to standardize the output of that response for easy parsing. 
- Return the final ranking

In [ ]:
# get the list of names from the dataframe
player_names = qual_df['Player'].tolist()
player_names = player_names[:35]
player_names

import ast

def parse_notable_wins(df, player_name):
    """Extracts names from the 'Notable Wins' column for a given player, handling string representations of lists."""
    player_data = df[df['Player'] == player_name].iloc[0]
    # Evaluate the string as a list of tuples
    notable_wins = ast.literal_eval(player_data['Notable Wins'])
    # Extract only the first item of each tuple
    notable_wins = [win[0] for win in notable_wins]
    return notable_wins

def compare_notable_wins(df, player1, player2):
    """Compares notable wins of two players and returns common and unique names."""
    wins_p1 = parse_notable_wins(df, player1)
    wins_p2 = parse_notable_wins(df, player2)
    
    # Sets for comparison
    wins_p1_set = set(wins_p1)
    wins_p2_set = set(wins_p2)
    
    # Calculate common and unique names
    common_wins = list(wins_p1_set & wins_p2_set)
    unique_wins_p1 = list(wins_p1_set - wins_p2_set)
    unique_wins_p2 = list(wins_p2_set - wins_p1_set)
    
    return common_wins, unique_wins_p1, unique_wins_p2

both_wins, p1_wins, p2_wins = compare_notable_wins(qual_df, 'Nessboy12', 'Dtier')

print(both_wins)
print(p1_wins)
print(p2_wins)

def parse_notable_losses(df, player_name):
    player_data = df[df['Player'] == player_name].iloc[0]
    # Evaluate the string as a list of tuples
    notable_losses = ast.literal_eval(player_data['Notable Losses'])
    # Extract only the first item of each tuple
    notable_losses = [loss[0] for loss in notable_losses]
    return notable_losses

def compare_notable_losses(df, player1, player2):
    losses_p1 = parse_notable_losses(df, player1)
    losses_p2 = parse_notable_losses(df, player2)
    
    # Sets for comparison
    losses_p1_set = set(losses_p1)
    losses_p2_set = set(losses_p2)
    
    # Calculate common and unique names
    common_losses = list(losses_p1_set & losses_p2_set)
    unique_losses_p1 = list(losses_p1_set - losses_p2_set)
    unique_losses_p2 = list(losses_p2_set - losses_p1_set)
    
    return common_losses, unique_losses_p1, unique_losses_p2

both_losses, p1_losses, p2_losses = compare_notable_losses(qual_df, 'Nessboy12', 'Widara')

print(both_losses)
print(p1_losses)
print(p2_losses)

def get_names_and_elos(df, names):
    """Takes a list of names and a DataFrame, returns a list of tuples (name, ELO)."""
    # Filter the DataFrame to only include rows with names that are in the provided list
    filtered_df = df[df['Player'].isin(names)]
    # Create a list of tuples from the filtered DataFrame
    name_elo_pairs = list(zip(filtered_df['Player'], filtered_df['ELO']))
    return name_elo_pairs

elo_list = get_names_and_elos(qual_df, p1_losses)
print(elo_list)

def loss_to_tournament_ratio(df, name):
    """Calculate the loss to tournament ratio for a given player."""
    player_data = df[df['Player'] == name].iloc[0]
    notable_losses = len(player_data['Notable Losses'])  # Assuming 'Notable Losses' is a list of losses
    tournaments_attended = player_data['Attendance']
    if tournaments_attended == 0:  # Avoid division by zero
        return float('inf')  # Infinite ratio if no tournaments attended
    return notable_losses / tournaments_attended

def compare_players_loss_ratio(df, player1, player2):
    """Compares the loss to tournament ratios of two players."""
    ratio1 = loss_to_tournament_ratio(df, player1)
    ratio2 = loss_to_tournament_ratio(df, player2)
    return ratio1, ratio2

ratio1, ratio2 = compare_players_loss_ratio(qual_df, 'Widara', 'Nessboy12')
print(ratio1, ratio2)

p1 = 'Widara'
p2 = 'Nessboy12'

both_wins, p1_wins, p2_wins = compare_notable_wins(qual_df, p1, p2)
both_wins = get_names_and_elos(qual_df, both_wins)
p1_wins = get_names_and_elos(qual_df, p1_wins)
p2_wins = get_names_and_elos(qual_df, p2_wins)

print(both_wins)
print(p1_wins)
print(p2_wins)

def calculate_elo(rating1, elo1, rating2, elo2, rating_oor):
    # Calculate coefficients a and b
    a = (elo1 - elo2) / (rating1 - rating2)
    b = elo1 - (a * rating1)
    # Calculate ELO for oor player
    elo_oor = a * rating_oor + b
    return elo_oor

# need to add OOR info to player cards

both_ls, p1_ls, p2_ls = compare_notable_losses(qual_df, p1, p2)
both_ls = get_names_and_elos(new_df, both_ls)
p1_ls = get_names_and_elos(new_df, p1_ls)
p2_ls = get_names_and_elos(new_df, p2_ls)

print(both_ls)
print(p1_ls)
print(p2_ls)

ratio1, ratio2 = compare_players_loss_ratio(qual_df, p1, p2)
print(ratio1, ratio2)

player_card = {
    'Players': [p1, p2],
    'Shared Wins': both_wins,
    'Unique Wins': [p1_wins, p2_wins],
    'Shared Losses': both_ls,
    'Unique Losses': [p1_ls, p2_ls],
    'LT Ratio': [ratio1, ratio2] 
}

player_card

from openai import OpenAI, completions

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# prompt could be improved
content = f'''
(Text Delimiters Given By ***)
***
Context: As a statistician, create an eSports power ranking based on player performance data provided below.
***
Objective: Decide which player is better based on wins, losses, and loss to tournament ratio. Higher ELO scores indicate better performance. A lower loss to tournament ratio is preferred.
***
Style: Return only the name of the better player, or 'Too Close' if a decision can't be reasonably made.
***
Response Format: Directly return the name of the better player. For example, if comparing 'John' and 'Sam' and John is deemed better, return 'John'. Do not return additional text or explanations.
***
Additional Context: Comparing two players:
- Player 1: {player_card['Players'][0]}
- Player 2: {player_card['Players'][1]}
Shared Wins: {player_card['Shared Wins']}
Unique Wins:
- {player_card['Players'][0]}: {player_card['Unique Wins'][0]}
- {player_card['Players'][1]}: {player_card['Unique Wins'][1]}
Shared Losses: {player_card['Shared Losses']}
Unique Losses:
- {player_card['Players'][0]}: {player_card['Unique Losses'][0]}
- {player_card['Players'][1]}: {player_card['Unique Losses'][1]}
Loss to Tournament Ratio:
- {player_card['Players'][0]}: {player_card['LT Ratio'][0]}
- {player_card['Players'][1]}: {player_card['LT Ratio'][1]}
'''

# response = client.chat.completions.create(
#     model="gpt-3.5-turbo",
#     messages=[
#         {"role": "system", "content": "You are a statistician who creates rankings."},
#         {"role": "user", "content": content}
#     ]
# )

# print(response.choices[0].message.content)

# the list we use for our top 5 players
player_names
test = player_names[:20]
test
elolist = get_names_and_elos(qual_df, test)
elolist

def group_by_ranking_distance(elo_list, window=5):
    """
    Groups players into overlapping windows based on ranking.
    
    Parameters:
    elo_list (list of tuples): List of (player, ELO) sorted by ELO (highest first).
    window (int): How many adjacent players each player should be compared to.
    
    Returns:
    list of lists: Groups of players that should be compared.
    """
    groups = []
    
    for i in range(len(elo_list)):
        # Create a group for each player containing themselves and the next 'window' players
        group = [elo_list[j][0] for j in range(i, min(i + window, len(elo_list)))]
        groups.append(group)
    
    return groups

# Sort by ELO (highest first)
elo_list_sorted = sorted(elolist, key=lambda x: x[1], reverse=True)

# Group players by ranking distance
groups = group_by_ranking_distance(elo_list_sorted, window=5)

groups

def parse_h2h_data(df, player_name, column_name):
    """General function to parse H2H data based on column name."""
    player_data = df[df['Player'] == player_name].iloc[0]
    h2h_data = ast.literal_eval(player_data[column_name])
    # Return only the names from the tuples (if tuples are used similar to wins/losses)
    return [data[0] for data in h2h_data] if h2h_data else []

def compare_h2h(df, player1, player2, column_name):
    """Compares H2H data of two players based on the specified column and returns common and unique names."""
    h2h_p1 = parse_h2h_data(df, player1, column_name)
    h2h_p2 = parse_h2h_data(df, player2, column_name)
    
    # Sets for comparison
    h2h_p1_set = set(h2h_p1)
    h2h_p2_set = set(h2h_p2)
    
    # Calculate common and unique names
    common_h2h = list(h2h_p1_set & h2h_p2_set)
    unique_h2h_p1 = list(h2h_p1_set - h2h_p2_set)
    unique_h2h_p2 = list(h2h_p2_set - h2h_p1_set)
    
    return common_h2h, unique_h2h_p1, unique_h2h_p2

def generate_player_card(df, player1, player2):
    both_wins, p1_wins, p2_wins = compare_notable_wins(df, player1, player2)
    both_wins = get_names_and_elos(df, both_wins)
    p1_wins = get_names_and_elos(df, p1_wins)
    p2_wins = get_names_and_elos(df, p2_wins)

    both_losses, p1_losses, p2_losses = compare_notable_losses(df, player1, player2)
    both_losses = get_names_and_elos(df, both_losses)
    p1_losses = get_names_and_elos(df, p1_losses)
    p2_losses = get_names_and_elos(df, p2_losses)

    ratio1, ratio2 = compare_players_loss_ratio(df, player1, player2)

    # Comparing Pos H2H, Even H2H, and Neg H2H
    pos_h2h_common, pos_h2h_unique_p1, pos_h2h_unique_p2 = compare_h2h(df, player1, player2, 'Pos H2H')
    even_h2h_common, even_h2h_unique_p1, even_h2h_unique_p2 = compare_h2h(df, player1, player2, 'Even H2H')
    neg_h2h_common, neg_h2h_unique_p1, neg_h2h_unique_p2 = compare_h2h(df, player1, player2, 'Neg H2H')

    player_card = {
        'Players': [player1, player2],
        'Shared Wins': both_wins,
        'Unique Wins': [p1_wins, p2_wins],
        'Shared Losses': both_losses,
        'Unique Losses': [p1_losses, p2_losses],
        'LT Ratio': [ratio1, ratio2],
        'Positive H2H': {
            'Common': pos_h2h_common,
            'Unique to Player 1': pos_h2h_unique_p1,
            'Unique to Player 2': pos_h2h_unique_p2
        },
        'Even H2H': {
            'Common': even_h2h_common,
            'Unique to Player 1': even_h2h_unique_p1,
            'Unique to Player 2': even_h2h_unique_p2
        },
        'Negative H2H': {
            'Common': neg_h2h_common,
            'Unique to Player 1': neg_h2h_unique_p1,
            'Unique to Player 2': neg_h2h_unique_p2
        }
    }

    return player_card

def get_ai_decision(player_card):
    content_template = f'''
    (Text Delimiters Given By ***)
    ***
    Context: As a statistician, create an eSports power ranking based on player performance data provided below.
    ***
    Objective: Decide which player is better based on wins, losses, loss to tournament ratio, and head-to-head comparisons. Higher ELO scores indicate a better player. A lower loss to tournament ratio is preferred. Head-to-head records provide insights into direct competition outcomes. Evaluate based on the following criteria:
    - Prioritize higher quality wins.
    - Penalize higher quantities of bad losses.
    - Reward players who have wins over individuals that their opponent has lost to.
    - Value consistency in performance, especially in terms of maintaining fewer losses and achieving higher peaks.
    - Consider head-to-head data:
    - Positive H2H indicates a direct advantage.
    - Even H2H shows parity between players.
    - Negative H2H suggests a disadvantage.
    ***
    Style: Return only the name of the better player. You must make a decision.
    ***
    Response Format: Directly return the name of the better player. For example, if comparing 'John' and 'Sam' and John is deemed better, return 'John'. Do not return additional text or explanations.
    ***
    Additional Context: Comparing two players:
    - Player 1: {player_card['Players'][0]}
    - Player 2: {player_card['Players'][1]}
    Shared Wins: {player_card['Shared Wins']}
    Unique Wins:
    - {player_card['Players'][0]}: {player_card['Unique Wins'][0]}
    - {player_card['Players'][1]}: {player_card['Unique Wins'][1]}
    Shared Losses: {player_card['Shared Losses']}
    Unique Losses:
    - {player_card['Players'][0]}: {player_card['Unique Losses'][0]}
    - {player_card['Players'][0]}: {player_card['Unique Losses'][1]}
    Loss to Tournament Ratio:
    - {player_card['Players'][0]}: {player_card['LT Ratio'][0]}
    - {player_card['Players'][1]}: {player_card['LT Ratio'][1]}
    Positive H2H:
    - Common: {player_card['Positive H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Positive H2H']['Unique to Player 1']}
    - U{player_card['Players'][0]}: {player_card['Positive H2H']['Unique to Player 2']}
    Even H2H:
    - Common: {player_card['Even H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Even H2H']['Unique to Player 1']}
    - {player_card['Players'][0]}: {player_card['Even H2H']['Unique to Player 2']}
    Negative H2H:
    - Common: {player_card['Negative H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Negative H2H']['Unique to Player 1']}
    - {player_card['Players'][0]}: {player_card['Negative H2H']['Unique to Player 2']}
    '''

    decisions = []
    for _ in range(3):  # Run the decision 5 times
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[
                {"role": "system", "content": "You are a statistician who creates rankings."},
                {"role": "user", "content": content_template}
            ]
        )
        decision = response.choices[0].message.content.strip()
        decisions.append(decision)
        # print(f"AI decision received: {decision}")

    # Count the frequency of each decision
    decision_count = {}
    for decision in decisions:
        if decision in decision_count:
            decision_count[decision] += 1
        else:
            decision_count[decision] = 1

    # Determine the most frequent decision
    max_decision = max(decision_count, key=decision_count.get)
    max_count = decision_count[max_decision]

    # Check if there is a clear winner or if it's too close
    if max_count > len(decisions) / 2:  # More than half the votes
        print(f"Most frequent decision: {max_decision} with {max_count} out of 3 votes")
        return max_decision
    else:
        print("Decision is too close to call")
        return 'Too Close'

    


In [ ]:
def ai_justification(player_card, decision):
    content_template = f'''
    (Text Delimiters Given By ***)
    ***
    Context: As a statistician, you are justifying your decision based on the criteria below. The player you decided was better is {decision}.
    ***
    Objective: You have already decided which player is better based on wins, losses, and loss to tournament ratio. Higher ELO scores indicate a better player. A lower loss to tournament ratio is preferred. 
    Your general criteria is to first evaluate wins, and then check losses. 
    - If people have similar quality wins, then the person with less bad losses should be valued.
    - If two players both have many bad losses, then the person with higher peaks should be valued. 
    - If two people are similar in terms of wins, then the person with better consistency in terms of wins and losses should be ranked higher
    - A significantly higher raw number of losses should be penalized
    - If one player has wins on the people that the other player has lost to, that should be rewarded
    ***
    Style: Create 4 sentence justification as to why you ranked {decision} higher. Keep it statistical in nature, and explain who is better based on the criteria.
    ***
    Response Format: A 4 sentence justification as to why you ranked {decision} higher.
    ***
    Additional Context: Comparing two players:
    - Player 1: {player_card['Players'][0]}
    - Player 2: {player_card['Players'][1]}
    Shared Wins: {player_card['Shared Wins']}
    Unique Wins:
    - {player_card['Players'][0]}: {player_card['Unique Wins'][0]}
    - {player_card['Players'][1]}: {player_card['Unique Wins'][1]}
    Shared Losses: {player_card['Shared Losses']}
    Unique Losses:
    - {player_card['Players'][0]}: {player_card['Unique Losses'][0]}
    - {player_card['Players'][1]}: {player_card['Unique Losses'][1]}
    Loss to Tournament Ratio:
    - {player_card['Players'][0]}: {player_card['LT Ratio'][0]}
    - {player_card['Players'][1]}: {player_card['LT Ratio'][1]}
    Positive H2H:
    - Common: {player_card['Positive H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Positive H2H']['Unique to Player 1']}
    - U{player_card['Players'][0]}: {player_card['Positive H2H']['Unique to Player 2']}
    Even H2H:
    - Common: {player_card['Even H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Even H2H']['Unique to Player 1']}
    - {player_card['Players'][0]}: {player_card['Even H2H']['Unique to Player 2']}
    Negative H2H:
    - Common: {player_card['Negative H2H']['Common']}
    - {player_card['Players'][0]}: {player_card['Negative H2H']['Unique to Player 1']}
    - {player_card['Players'][0]}: {player_card['Negative H2H']['Unique to Player 2']}
    '''

    response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[
                {"role": "system", "content": "You are a statistician who creates rankings."},
                {"role": "user", "content": content_template}
            ]
        )
    
    justification = response.choices[0].message.content.strip()
    return justification


In [ ]:
def compare_players(player_list, df):
    if not player_list:
        return []  # Return an empty list if the input list is empty

    # Initialize ranked_players with the first player
    ranked_players = [player_list[0]]
    
    for player in player_list[1:]:
        inserted = False
        for i in range(len(ranked_players)):
            print(f"Comparing {ranked_players[i]} and {player}")
            player_card = generate_player_card(df, ranked_players[i], player)
            while True:
                better_player = get_ai_decision(player_card)
                # justification = ai_justification(player_card, better_player)
                # print(justification)
                if better_player == ranked_players[i] or better_player == player:
                    if better_player == player:
                        # Insert the new player right before the player they defeated
                        ranked_players.insert(i, player)
                        inserted = True
                    break
                else:
                    print("Invalid entry, please enter one of the two names exactly as shown.")
            if inserted:
                break
        if not inserted:
            # If no one was considered better than the new player, append them to the end
            ranked_players.append(player)

    return ranked_players

In [ ]:
# from collections import defaultdict

# final_scores = defaultdict(int)  # Dictionary to track player scores (higher score = higher rank)
# seen_pairs = set()  # Cache for compared player pairs

# # Iterate through each sublist (group) of players
# for group_num, group in enumerate(groups, start=1):  # Track group number
#     if len(group) > 1:  # Only process groups with at least two players
#         print(f"\n🔹 Processing group {group_num}/{len(groups)}: {group}")  # Progress update
        
#         valid_comparisons = []  # Store valid player pairs
        
#         for i in range(len(group)):
#             for j in range(i + 1, len(group)):  # Only compare unique pairs
#                 player1, player2 = group[i], group[j]
                
#                 if player1 == player2:  # 🚨 Fix: Prevent self-comparisons
#                     continue  
                
#                 pair = tuple(sorted([player1, player2]))  # Order-independent key
                
#                 if pair not in seen_pairs:  # Ensure comparison hasn't been made before
#                     valid_comparisons.append(pair)
#                     seen_pairs.add(pair)  # Mark as compared

#         if valid_comparisons:  # Only call compare_players if new comparisons exist
#             for player1, player2 in valid_comparisons:  # Compare pairs explicitly
#                 print(f"\n⚔️ Comparing {player1} vs {player2}...")  # Show comparison
#                 ranking = compare_players([player1, player2], new_df)  
                
#                 winner = ranking[0]  # The first player in `ranking` is the winner
#                 final_scores[winner] += 1  
                
#                 print(f"🏆 Winner: {winner} | Updated Score: {final_scores[winner]}")
#                 print(f"🔄 Current Scoreboard: {dict(final_scores)}")  # Show ongoing scores

# # Sort players by highest score
# sorted_rankings = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

# # Print the final ranking
# print("\n📊 FINAL RANKING:")
# for rank, (player, score) in enumerate(sorted_rankings, start=1):
#     print(f"{rank}. {player} ({score} wins)")

In [ ]:
# def rank_players(groups, df, max_elo_gap=100):  # Limit comparisons to players with similar ELOs
#     ranked_players = []  # Ordered list of players
#     seen_pairs = set()  # Track which comparisons have been made

#     def get_elo(player, df):
#         """Fetch ELO for a player from the dataframe."""
#         return df.loc[df["Player"] == player, "ELO"].values[0] if player in df["Player"].values else 0

#     # Iterate through each group in order
#     for group_num, group in enumerate(groups, start=1):
#         if len(group) > 1:  # Only process meaningful groups
#             print(f"\n🔹 Processing group {group_num}/{len(groups)}: {group}")  # Progress update

#             for player in group:
#                 if player not in ranked_players:
#                     # If the player isn't in the ranking yet, insert them in the correct place
#                     inserted = False
#                     player_elo = get_elo(player, df)

#                     for i in range(len(ranked_players)):
#                         ranked_opponent = ranked_players[i]
#                         opponent_elo = get_elo(ranked_opponent, df)

#                         pair = tuple(sorted([player, ranked_opponent]))  # Ensure order-independence

#                         # 🚨 **NEW RULE: Only compare if ELO difference is within threshold**
#                         if abs(player_elo - opponent_elo) > max_elo_gap:
#                             continue  # Skip comparisons if too far apart in skill

#                         if pair in seen_pairs:  # Skip redundant comparisons
#                             continue
                        
#                         print(f"\n⚔️ Comparing {player} vs {ranked_opponent}...")  # Show comparison
#                         ranking = compare_players([player, ranked_opponent], df)

#                         better_player = ranking[0]  # The first player in `ranking` is the winner
#                         seen_pairs.add(pair)  # Mark this comparison as done

#                         if better_player == player:
#                             ranked_players.insert(i, player)  # Insert player before their weaker opponent
#                             inserted = True
#                             break  # Stop inserting since the correct spot is found

#                     if not inserted:
#                         ranked_players.append(player)  # Append to end if they never got inserted

#     return ranked_players


# # Run the ranking system
# final_ranking = rank_players(groups, new_df, max_elo_gap=100)  # Only compare players within 100 ELO

# # Print the final ordered ranking
# print("\n📊 FINAL RANKING:")
# for rank, player in enumerate(final_ranking, start=1):
#     print(f"{rank}. {player}")